# Data Exploration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

In [2]:
df_weather = pd.read_csv(r'../Cleaned Data/weather_cleaned.csv')
df_tramway_Debourg = pd.read_csv(r'../Cleaned Data/tramway_1_direction_Debourg_1.csv')
df_tramway_IUT_Feyssine = pd.read_csv(r'../Cleaned Data/tramway_1_direction_IUT_Feyssine_2.csv')

In [6]:
print(df_weather.columns)
df_weather.head()

Index(['VAL_DATE', 'Temperature', 'weather_code', 'Condition',
       'Associated number'],
      dtype='str')


,VAL_DATE,Temperature,weather_code,Condition,Associated number
0,1/1/19 00:00,2,266,Light drizzle,1
1,1/1/19 00:15,2,266,Light drizzle,1
2,1/1/19 00:30,2,266,Light drizzle,1
3,1/1/19 00:45,2,266,Light drizzle,1
4,1/1/19 01:00,2,266,Light drizzle,1


In [7]:
print(df_tramway_Debourg.columns)
df_tramway_Debourg.head()

Index(['VAL_ARRET_CODE', 'VAL_DATE_Hour', 'CRS_SENS_TRAJET', 'Flow',
       'Day of week', 'Day of month', 'Month', 'Hour', 'Is_workday',
       'Is_holiday', 'Is_peak_hour'],
      dtype='str')


,VAL_ARRET_CODE,VAL_DATE_Hour,CRS_SENS_TRAJET,Flow,Day of week,Day of month,Month,Hour,Is_workday,Is_holiday,Is_peak_hour
0,32102,2019-11-01 00:00:00,1,55,4,1,11,0,1,0,0
1,32102,2019-11-01 05:00:00,1,17,4,1,11,5,1,0,0
2,32102,2019-11-01 06:00:00,1,25,4,1,11,6,1,0,0
3,32102,2019-11-01 07:00:00,1,30,4,1,11,7,1,0,1
4,32102,2019-11-01 08:00:00,1,53,4,1,11,8,1,0,1


In [8]:
print(df_tramway_IUT_Feyssine.columns)
df_tramway_IUT_Feyssine.head()

Index(['VAL_ARRET_CODE', 'VAL_DATE_Hour', 'CRS_SENS_TRAJET', 'Flow',
       'Day of week', 'Day of month', 'Month', 'Hour', 'Is_workday',
       'Is_holiday', 'Is_peak_hour'],
      dtype='str')


,VAL_ARRET_CODE,VAL_DATE_Hour,CRS_SENS_TRAJET,Flow,Day of week,Day of month,Month,Hour,Is_workday,Is_holiday,Is_peak_hour
0,32103,2019-11-01 00:00:00,2,24,4,1,11,0,1,0,0
1,32103,2019-11-01 05:00:00,2,19,4,1,11,5,1,0,0
2,32103,2019-11-01 06:00:00,2,26,4,1,11,6,1,0,0
3,32103,2019-11-01 07:00:00,2,40,4,1,11,7,1,0,1
4,32103,2019-11-01 08:00:00,2,67,4,1,11,8,1,0,1


## Data Patterns

### Jour type et semaine type

In [ ]:
# Flux total de la ligne par créneau de 15 min
flux = df_t1.groupby('VAL_DATE')['Flow'].sum()

# Grille complète de créneaux (jours entiers) : créneau absent = 0 validation
grille = pd.date_range(flux.index.min().normalize(),
                       flux.index.max().normalize() + pd.Timedelta(hours=23, minutes=45),
                       freq='15min')
flux = flux.reindex(grille, fill_value=0).rename('Flow').to_frame()

flux['date'] = flux.index.normalize()
flux['heure'] = flux.index.hour + flux.index.minute / 60
flux['jour_semaine'] = flux.index.weekday
flux['type_jour'] = np.select([flux['jour_semaine'] < 5, flux['jour_semaine'] == 5],
                              ['Lundi–vendredi', 'Samedi'], 'Dimanche')
flux['heure_semaine'] = flux['jour_semaine'] * 24 + flux['heure']
flux['debut_semaine'] = flux['date'] - pd.to_timedelta(flux['jour_semaine'], unit='D')
flux.head()

### Jour type pour chaque jour de la semaine

In [ ]:
profil_js = flux.pivot_table(index='heure', columns='jour_semaine', values='Flow', aggfunc='mean')
profil_js.columns = [JOURS[j] for j in profil_js.columns]

ax = profil_js.plot(figsize=(12, 5), lw=1.8, cmap='tab10')
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2))
ax.set_xlabel('Heure')
ax.set_ylabel('Validations par 15 min (moyenne)')
ax.set_title('Profil horaire moyen par jour de la semaine — ligne T1')
ax.grid(alpha=0.3)
plt.show()

### Semaine type (chaque semaine en gris, moyenne en couleur)

In [ ]:
# On ne garde que les semaines complètes (lundi -> dimanche)
nb_jours = flux.groupby('debut_semaine')['date'].nunique()
semaines_completes = nb_jours[nb_jours == 7].index
fs = flux[flux['debut_semaine'].isin(semaines_completes)]

semaines = fs.pivot_table(index='heure_semaine', columns='debut_semaine', values='Flow')
moyenne = semaines.mean(axis=1)

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(moyenne.index, moyenne.values, color='tab:red', lw=2, label='Semaine type (moyenne)')

for j in range(1, 7):
    ax.axvline(24 * j, color='black', lw=0.6, alpha=0.4)
ax.set_xticks([24 * j + 12 for j in range(7)])
ax.set_xticklabels(JOURS)
ax.set_xlim(0, 168)
ax.set_ylabel('Validations par 15 min')
ax.set_title(f'Semaine type — ligne T1 ({len(semaines_completes)} semaines complètes)')
ax.legend(loc='upper right')
ax.grid(alpha=0.3, axis='y')
plt.show()

### Fréquentation journalière : repérer les semaines atypiques (vacances, grèves...)

In [ ]:
total_jour = flux.groupby('date')['Flow'].sum()

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(total_jour.index, total_jour.values, marker='o', ms=3, lw=1)
weekend = total_jour[total_jour.index.weekday >= 5]
ax.scatter(weekend.index, weekend.values, color='tab:orange', s=12, zorder=3, label='Week-end')
ax.set_ylabel('Validations par jour')
ax.set_title('Validations journalières — ligne T1')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## First hypotheses

Hypothèses :
- On enlève la période COVID
- On étudie la ligne T1
- On split le data set en deux : le training set [2019-11-01; 2019-12-15] et le validation set [2020-01-15; 2020-03-15] (environ)
- On enlève les jours avec une valeur de flow abhérante : flow = 0
- Dans les deux sens
- Validation toutes les 15min

AI Help
- Le fichier ne contient une ligne que lorsqu'il y a au moins une validation : les créneaux absents valent donc 0. On reconstruit une grille complète de créneaux avant de moyenner, sinon les heures creuses sont surestimées (c'est aussi pourquoi la moyenne de `Flow` ligne par ligne, plus haut, est biaisée).

**Il faut verifier les incohérences**